# js

> JavaScript generation functions for viewport height measurement

In [ ]:
#| default_exp js

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from cjm_fasthtml_viewport_fit.models import ViewportFitConfig

## Debug Helpers

Conditional logging controlled by a window-level debug flag.

In [ ]:
#| export
def generate_debug_helpers_js(
    config: ViewportFitConfig,  # Instance configuration
) -> str:  # JS debug helper closures
    """Generate debug logging helpers conditional on a window flag."""
    enable_line = f"window.{config.debug_flag} = true;" if config.debug else ""
    return f"""
    {enable_line}
    const _debug = () => window.{config.debug_flag} === true;
    const _log = (...args) => {{ if (_debug()) console.log('[{config.log_prefix}]', ...args); }};
"""

## Space Below Measurement

Walks the DOM tree upward from the target element, measuring sibling elements positioned below and accumulating parent padding/borders. Uses position-based detection to correctly handle flex/grid rows where siblings may be beside rather than below.

In [ ]:
#| export
def generate_space_below_js(
    config: ViewportFitConfig,  # Instance configuration
) -> str:  # JS function for DOM-walking space measurement
    """Generate JS that walks the DOM tree to measure space below a target element."""
    return f"""
    function _calculateSpaceBelow(target) {{
        let spaceBelow = 0;
        let currentElement = target;
        let currentRect = currentElement.getBoundingClientRect();
        let parent = target.parentElement;
        let level = 0;

        while (parent && parent !== document.documentElement) {{
            const parentStyles = getComputedStyle(parent);
            const paddingBottom = parseFloat(parentStyles.paddingBottom) || 0;
            const borderBottom = parseFloat(parentStyles.borderBottomWidth) || 0;
            const marginBottom = parseFloat(parentStyles.marginBottom) || 0;

            _log(`spaceBelow L${{level}}: parent=${{parent.tagName}}#${{parent.id || '(no id)'}}, padding=${{paddingBottom}}, border=${{borderBottom}}, margin=${{marginBottom}}`);

            let foundCurrent = false;
            for (const sibling of parent.children) {{
                if (sibling === currentElement) {{ foundCurrent = true; continue; }}
                if (!foundCurrent) continue;
                if (sibling.nodeType !== Node.ELEMENT_NODE) continue;
                const tag = sibling.tagName;
                if (tag === 'SCRIPT' || tag === 'STYLE') continue;
                if (tag === 'INPUT' && sibling.type === 'hidden') continue;
                const s = getComputedStyle(sibling);
                if (s.display === 'none') continue;

                const siblingRect = sibling.getBoundingClientRect();

                // Skip siblings beside (not below) in flex/grid rows
                if (siblingRect.top < currentRect.bottom) {{
                    _log(`  skip (beside): ${{tag}}#${{sibling.id || '(no id)'}}, top=${{siblingRect.top}} < bottom=${{currentRect.bottom}}`);
                    continue;
                }}

                const visualSpace = siblingRect.bottom - currentRect.bottom;
                spaceBelow += visualSpace;
                _log(`  below: ${{tag}}#${{sibling.id || '(no id)'}}, visualSpace=${{visualSpace}}`);

                currentRect = {{ bottom: siblingRect.bottom, top: currentRect.top }};
            }}

            spaceBelow += paddingBottom + borderBottom + marginBottom;

            currentElement = parent;
            currentRect = parent.getBoundingClientRect();
            parent = parent.parentElement;
            level++;
        }}
        return spaceBelow;
    }}
"""

## Height Calculation

Temporarily collapses the target element to 0px to measure its natural top position, then calculates the remaining viewport height after accounting for space above and below.

When the target has padding/border, collapsing to `height: 0` still leaves a non-zero rendered box. The algorithm accounts for this by measuring the collapsed box height and adjusting based on `box-sizing`:
- **content-box** (default): applied height is added ON TOP of padding+border, so subtract the collapsed height
- **border-box**: applied height INCLUDES padding+border, so use the raw measurement

In [ ]:
#| export
def generate_calculate_height_js(
    config: ViewportFitConfig,  # Instance configuration
) -> str:  # JS function for collapse-measure-apply height calculation
    """Generate JS that measures and applies viewport-fitted height to the target element."""
    # Inner element handling (optional)
    if config.inner_id:
        inner_lookup = f"const inner = document.getElementById('{config.inner_id}');"
        inner_collapse = "if (inner) inner.style.height = '0px';"
        inner_apply = "if (inner) inner.style.height = viewportHeight + 'px';"
    else:
        inner_lookup = ""
        inner_collapse = ""
        inner_apply = ""

    return f"""
    function _calculateAndSetHeight() {{
        const target = document.getElementById('{config.target_id}');
        {inner_lookup}
        if (!target) {{
            _log('ERROR: target element not found:', '{config.target_id}');
            return {config.min_height};
        }}

        _log('=== _calculateAndSetHeight ===');

        // Save current height
        const savedHeight = target.style.height;
        const savedMinHeight = target.style.minHeight;
        const savedMaxHeight = target.style.maxHeight;

        // Temporarily collapse to measure natural top position
        target.style.height = '0px';
        target.style.minHeight = '0px';
        target.style.maxHeight = '0px';
        {inner_collapse}

        // Force reflow
        target.offsetHeight;

        // Measure positions with collapsed target
        const targetRect = target.getBoundingClientRect();
        const spaceAbove = targetRect.top;
        const spaceBelow = _calculateSpaceBelow(target);

        // Collapsed target may still have non-zero height from padding/border.
        // For content-box, the applied height is ON TOP of padding/border,
        // so we must subtract the collapsed box height to avoid overflow.
        // For border-box, the applied height INCLUDES padding/border.
        const collapsedHeight = targetRect.bottom - targetRect.top;
        const targetBoxSizing = getComputedStyle(target).boxSizing;
        const boxAdjust = targetBoxSizing === 'border-box' ? 0 : collapsedHeight;

        const windowHeight = window.innerHeight;
        const rawHeight = windowHeight - spaceAbove - spaceBelow - boxAdjust;
        const viewportHeight = Math.floor(Math.max({config.min_height}, rawHeight));

        _log('windowHeight:', windowHeight);
        _log('spaceAbove:', spaceAbove);
        _log('spaceBelow:', spaceBelow);
        _log('collapsedHeight:', collapsedHeight, '(boxSizing:', targetBoxSizing + ', adjust:', boxAdjust + ')');
        _log('rawHeight:', rawHeight);
        _log('viewportHeight:', viewportHeight, rawHeight < {config.min_height} ? '(clamped to min)' : '');

        // Apply calculated height
        target.style.height = viewportHeight + 'px';
        target.style.maxHeight = viewportHeight + 'px';
        target.style.minHeight = viewportHeight + 'px';
        {inner_apply}

        return viewportHeight;
    }}
"""

## Resize Handler

Debounced window resize listener with cleanup for HTMX re-execution.

In [ ]:
#| export
def generate_resize_handler_js(
    config: ViewportFitConfig,  # Instance configuration
) -> str:  # JS for debounced resize listener with cleanup
    """Generate debounced window resize handler with HTMX-safe cleanup."""
    callback_line = f"\n        {config.resize_callback}" if config.resize_callback else ""

    return f"""
    function _debounce(fn, delay) {{
        let tid;
        return function(...args) {{ clearTimeout(tid); tid = setTimeout(() => fn.apply(this, args), delay); }};
    }}

    const _debouncedResize = _debounce(function() {{
        _calculateAndSetHeight();{callback_line}
    }}, {config.debounce_ms});

    if (window.{config.handler_key}) {{
        window.removeEventListener('resize', window.{config.handler_key});
    }}
    window.{config.handler_key} = _debouncedResize;
    window.addEventListener('resize', _debouncedResize);
"""

## HTMX Settle Handler

Remeasures on `htmx:afterSettle` events. Returns empty string if disabled.

In [ ]:
#| export
def generate_htmx_settle_js(
    config: ViewportFitConfig,  # Instance configuration
) -> str:  # JS for HTMX afterSettle recalculation (empty if disabled)
    """Generate HTMX afterSettle handler for recalculation on SPA navigation."""
    if not config.enable_htmx_settle:
        return ""

    # Re-setup sibling observer after HTMX swaps replace DOM elements
    observer_setup = "if (typeof _setupSiblingObserver === 'function') _setupSiblingObserver();" if config.observe_siblings else ""

    return f"""
    function _onSettle() {{
        requestAnimationFrame(function() {{
            {observer_setup}
            _calculateAndSetHeight();
        }});
    }}

    if (window.{config.settle_handler_key}) {{
        document.body.removeEventListener('htmx:afterSettle', window.{config.settle_handler_key});
    }}
    window.{config.settle_handler_key} = _onSettle;
    document.body.addEventListener('htmx:afterSettle', _onSettle);
"""

## Sibling ResizeObserver

Walks up the DOM tree from the target element and watches all visible siblings
at every ancestor level for size changes. This mirrors the DOM-walking pattern
of `_calculateSpaceBelow` — any element that contributes to the space
calculation is observed.

Runs `_calculateAndSetHeight()` synchronously in the ResizeObserver callback.
ResizeObserver fires between layout and paint, so the browser paints both the
sibling change and the target adjustment in the same frame — no visual
desync. A recursion guard prevents infinite loops if the target's height
change were to affect an observed element.

Returns empty string if disabled.

In [ ]:
#| export
def generate_sibling_observer_js(
    config: ViewportFitConfig,  # Instance configuration
) -> str:  # JS for ResizeObserver on ancestor-level siblings (empty if disabled)
    """Generate ResizeObserver that walks the DOM tree and watches visible siblings at all ancestor levels."""
    if not config.observe_siblings:
        return ""

    return f"""
    function _setupSiblingObserver() {{
        const target = document.getElementById('{config.target_id}');
        if (!target || !target.parentElement) return;

        // Disconnect previous observer (handles re-setup after HTMX swaps)
        if (window.{config.observer_key}) {{
            window.{config.observer_key}.disconnect();
        }}

        // Synchronous recalculation — ResizeObserver fires between layout
        // and paint, so modifying the target here lets the browser paint
        // both sibling and target changes in the same frame.
        let _recalculating = false;
        const _onResize = function() {{
            if (_recalculating) return;
            _recalculating = true;
            _log('layout change detected, recalculating');
            _calculateAndSetHeight();
            _recalculating = false;
        }};

        const observer = new ResizeObserver(_onResize);

        // Walk up the DOM tree mirroring _calculateSpaceBelow —
        // observe visible siblings at every ancestor level so that
        // changes anywhere in the layout trigger recalculation.
        let currentElement = target;
        let parent = target.parentElement;
        let level = 0;

        while (parent && parent !== document.documentElement) {{
            for (const sibling of parent.children) {{
                if (sibling === currentElement) continue;
                const tag = sibling.tagName;
                if (tag === 'SCRIPT' || tag === 'STYLE') continue;
                if (tag === 'INPUT' && sibling.type === 'hidden') continue;
                if (getComputedStyle(sibling).display === 'none') continue;
                observer.observe(sibling);
                _log('observing L' + level + ':', tag + '#' + (sibling.id || '(no id)'));
            }}
            currentElement = parent;
            parent = parent.parentElement;
            level++;
        }}

        window.{config.observer_key} = observer;
    }}
"""

## Initialization

Runs on first script execution with optional scroll-to-top for consistent HTMX SPA measurements.

In [ ]:
#| export
def generate_init_js(
    config: ViewportFitConfig,  # Instance configuration
) -> str:  # JS initialization block
    """Generate JS for initial measurement on script execution."""
    scroll_line = "window.scrollTo(0, 0);" if config.scroll_to_top else ""
    observer_setup = "_setupSiblingObserver();" if config.observe_siblings else ""

    return f"""
    requestAnimationFrame(function() {{
        setTimeout(function() {{
            {scroll_line}
            _calculateAndSetHeight();
            {observer_setup}
        }}, 50);
    }});
"""

## Top-Level Compositor

Assembles all JS fragments into a self-contained IIFE. Exposes a global recalculate function for manual invocation.

In [ ]:
#| export
def generate_viewport_fit_js(
    config: ViewportFitConfig,  # Instance configuration
) -> str:  # Complete JS code as a self-contained IIFE
    """Generate the complete viewport-fit JS as a self-contained IIFE."""
    parts = [
        generate_debug_helpers_js(config),
        generate_space_below_js(config),
        generate_calculate_height_js(config),
        generate_resize_handler_js(config),
        generate_htmx_settle_js(config),
        generate_sibling_observer_js(config),
        f"    window.{config.recalc_fn} = _calculateAndSetHeight;",
        generate_init_js(config),
    ]

    body = "\n".join(parts)
    return f"(function() {{{body}\n}})();"

## Tests

In [ ]:
from cjm_fasthtml_viewport_fit.models import ViewportFitConfig

# Default config
cfg = ViewportFitConfig(namespace="test", target_id="my-target")
js = generate_viewport_fit_js(cfg)

# Verify IIFE wrapper
assert js.startswith("(function() {")
assert js.endswith("})();")

# Verify core functions present
assert "_calculateSpaceBelow" in js
assert "_calculateAndSetHeight" in js
assert "_debounce" in js

# Verify target ID used
assert "'my-target'" in js

# Verify global recalculate function exposed
assert "window.recalculateTestHeight = _calculateAndSetHeight" in js

# Verify resize handler cleanup
assert "_vfResizeHandler_test" in js

# Verify HTMX settle handler present (enabled by default)
assert "htmx:afterSettle" in js
assert "_vfSettleHandler_test" in js

# Verify sibling ResizeObserver with ancestor-walking
assert "ResizeObserver" in js
assert "_vfSiblingObserver_test" in js
assert "_setupSiblingObserver" in js
# Verify ancestor-level walking (while loop + level tracking)
assert "while (parent && parent !== document.documentElement)" in js
assert "currentElement = parent" in js
assert "'observing L'" in js
# Verify synchronous recalculation with recursion guard
assert "_recalculating" in js
assert "_onResize" in js

# Verify scroll-to-top present (enabled by default)
assert "window.scrollTo(0, 0)" in js

# Verify min height used
assert "Math.max(200," in js

# Verify box-sizing adjustment for padding/border
assert "collapsedHeight" in js
assert "boxSizing" in js
assert "boxAdjust" in js

# Verify debug flag NOT enabled by default
assert "viewportFitDebug_test = true" not in js
assert "viewportFitDebug_test" in js  # but referenced for checking

print("Default config tests passed!")

In [ ]:
# Config with inner_id
cfg_inner = ViewportFitConfig(namespace="inner_test", target_id="t", inner_id="t-inner")
js_inner = generate_viewport_fit_js(cfg_inner)
assert "'t-inner'" in js_inner
assert "if (inner)" in js_inner

# Config without inner_id — no inner references
cfg_no_inner = ViewportFitConfig(namespace="no_inner", target_id="t")
js_no_inner = generate_viewport_fit_js(cfg_no_inner)
assert "'t-inner'" not in js_no_inner

print("Inner ID tests passed!")

Inner ID tests passed!


In [ ]:
# HTMX settle disabled
cfg_no_htmx = ViewportFitConfig(namespace="nohtmx", target_id="t", enable_htmx_settle=False)
js_no_htmx = generate_viewport_fit_js(cfg_no_htmx)
assert "htmx:afterSettle" not in js_no_htmx
assert "_vfSettleHandler" not in js_no_htmx

# Sibling observer disabled
cfg_no_obs = ViewportFitConfig(namespace="noobs", target_id="t", observe_siblings=False)
js_no_obs = generate_viewport_fit_js(cfg_no_obs)
assert "ResizeObserver" not in js_no_obs
assert "_vfSiblingObserver" not in js_no_obs

# Scroll-to-top disabled
cfg_no_scroll = ViewportFitConfig(namespace="noscroll", target_id="t", scroll_to_top=False)
js_no_scroll = generate_viewport_fit_js(cfg_no_scroll)
assert "window.scrollTo" not in js_no_scroll

# Debug enabled by default
cfg_debug = ViewportFitConfig(namespace="dbg", target_id="t", debug=True)
js_debug = generate_viewport_fit_js(cfg_debug)
assert "viewportFitDebug_dbg = true;" in js_debug

# Resize callback
cfg_cb = ViewportFitConfig(namespace="cb", target_id="t", resize_callback="myApp.onResize();")
js_cb = generate_viewport_fit_js(cfg_cb)
assert "myApp.onResize();" in js_cb

# Custom min height
cfg_min = ViewportFitConfig(namespace="min", target_id="t", min_height=300)
js_min = generate_viewport_fit_js(cfg_min)
assert "Math.max(300," in js_min

print("Feature flag tests passed!")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()